# WhatsApp to Excel - Manual Login Version

یہ version **manual login** کے ساتھ کام کرتا ہے۔

آپ **خود QR code scan** کریں گے۔


## Step 1: Install Packages

In [1]:
import subprocess
import sys

packages = ['selenium', 'openpyxl', 'webdriver-manager', 'pandas']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"OK {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"OK {package} installed")

print("\nOK All packages installed!")

OK selenium already installed
OK openpyxl already installed
OK webdriver-manager already installed
OK pandas already installed

OK All packages installed!


## Step 2: Import Libraries

In [2]:
import os
import time
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import openpyxl

print("OK All libraries imported!")

OK All libraries imported!


## Step 3: Define Automation Class

In [3]:
class WhatsAppManualLogin:
    def __init__(self, excel_file_path: str, start_date: str, end_date: str):
        self.excel_file_path = excel_file_path
        self.start_date = start_date
        self.end_date = end_date
        self.driver = None
        self.processed_messages = set()
        self.load_processed_messages()
        
    def load_processed_messages(self):
        cache_file = Path("processed_messages.json")
        if cache_file.exists():
            with open(cache_file, 'r') as f:
                self.processed_messages = set(json.load(f))
    
    def save_processed_messages(self):
        with open("processed_messages.json", 'w') as f:
            json.dump(list(self.processed_messages), f)
    
    def parse_date(self, date_str: str) -> datetime:
        """Parse date string like '1-4-2026' to datetime"""
        try:
            parts = date_str.split('-')
            day = int(parts[0])
            month = int(parts[1])
            year = int(parts[2])
            return datetime(year, month, day)
        except:
            return None
    
    def is_date_in_range(self, message_date_str: str) -> bool:
        """Check if message date is within the range"""
        msg_date = self.parse_date(message_date_str)
        start = self.parse_date(self.start_date)
        end = self.parse_date(self.end_date)
        
        if not msg_date or not start or not end:
            return False
        
        return start <= msg_date <= end
    
    def setup_driver(self):
        print("\n" + "="*60)
        print("Opening WhatsApp Web...")
        print("="*60 + "\n")
        
        chrome_options = Options()
        chrome_options.add_argument("--start-maximized")
        
        user_data_dir = str(Path.home() / ".whatsapp_manual_login")
        chrome_options.add_argument(f"user-data-dir={user_data_dir}")
        
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)
        
        self.driver.get("https://web.whatsapp.com")
        
        print("!"*60)
        print("MANUAL LOGIN:")
        print("1. Scan QR code with your phone")
        print("2. Open your complaint group")
        print("3. Press Enter below when ready")
        print("!"*60 + "\n")
        
        input("Press Enter when logged in and group is open...")
        
        print("\nOK Starting data collection...\n")
        time.sleep(2)
        return True
    
    def extract_messages(self) -> List[Dict]:
        messages = []
        
        try:
            message_elements = self.driver.find_elements(By.XPATH, "//div[@data-testid='msg-container']")
            
            for msg_elem in message_elements:
                try:
                    msg_text = msg_elem.text
                    
                    if not msg_text or len(msg_text) < 5:
                        continue
                    
                    msg_id = hash(msg_text) % ((2**31) - 1)
                    
                    if msg_id in self.processed_messages:
                        continue
                    
                    messages.append({
                        'id': msg_id,
                        'text': msg_text,
                        'timestamp': datetime.now().isoformat()
                    })
                    
                    self.processed_messages.add(msg_id)
                    
                except Exception:
                    continue
            
            return messages
            
        except Exception as e:
            print(f"Error extracting messages: {e}")
            return []
    
    def parse_complaint_data(self, message_text: str) -> Optional[Dict]:
        lines = message_text.strip().split('\n')
        
        if len(lines) < 4:
            return None
        
        data = {}
        
        # Extract time (Line 0)
        time_str = lines[0].strip()
        if time_str:
            data['Time'] = time_str
        else:
            return None
        
        # Extract date (Line 1)
        date_str = lines[1].strip()
        date_match = re.search(r'(\d{1,2})-(\d{1,2})-(\d{4})', date_str)
        if date_match:
            data['Date'] = date_match.group(0)
        else:
            return None
        
        # CHECK DATE RANGE
        if not self.is_date_in_range(data['Date']):
            return None
        
        # Extract branch (Line 2)
        branch_str = lines[2].strip()
        if 'Branch' in branch_str:
            data['Branch'] = branch_str.replace('Branch', '').strip()
        else:
            data['Branch'] = branch_str
        
        # Extract complaint (Line 3+)
        complaint_text = '\n'.join(lines[3:]).strip()
        if complaint_text:
            data['Complain'] = complaint_text[:200]
        else:
            return None
        
        # Categorize
        complaint_lower = complaint_text.lower()
        if 'staff' in complaint_lower or 'late' in complaint_lower or 'employee' in complaint_lower:
            data['Category'] = 'staff issue'
        elif 'food' in complaint_lower or 'quality' in complaint_lower or 'uncooked' in complaint_lower:
            data['Category'] = 'quality issue'
        elif 'service' in complaint_lower or 'slow' in complaint_lower or 'wait' in complaint_lower:
            data['Category'] = 'service issue'
        elif 'waste' in complaint_lower:
            data['Category'] = 'wastage issue'
        else:
            data['Category'] = 'other'
        
        data['Response (Y / N)'] = 'N'
        
        return data if len(data) > 2 else None
    
    def update_excel(self, complaint_data: Dict):
        try:
            if not os.path.exists(self.excel_file_path):
                print(f"ERROR Excel file not found: {self.excel_file_path}")
                return False
            
            wb = openpyxl.load_workbook(self.excel_file_path)
            ws = wb.active
            
            next_row = ws.max_row + 1
            
            columns = {
                'Date': 1,
                'Time': 2,
                'Branch': 3,
                'Category': 4,
                'Complain': 7,
                'Response (Y / N)': 8,
            }
            
            for key, col_num in columns.items():
                if key in complaint_data:
                    ws.cell(row=next_row, column=col_num, value=complaint_data[key])
            
            wb.save(self.excel_file_path)
            print(f"OK Added (Row {next_row}): {complaint_data.get('Complain', 'Unknown')[:40]}...")
            return True
            
        except Exception as e:
            print(f"ERROR updating Excel: {e}")
            return False
    
    def run_collection(self):
        """Collect data once"""
        if not self.setup_driver():
            return
        
        print(f"Collecting data from {self.start_date} to {self.end_date}...\n")
        
        try:
            for i in range(10):  # Check 10 times
                print(f"Scan #{i+1}...")
                
                messages = self.extract_messages()
                
                if messages:
                    print(f"  Found {len(messages)} new message(s)")
                    
                    for msg in messages:
                        complaint_data = self.parse_complaint_data(msg['text'])
                        
                        if complaint_data:
                            self.update_excel(complaint_data)
                else:
                    print("  No new messages")
                
                self.save_processed_messages()
                time.sleep(3)
        
        except KeyboardInterrupt:
            print("\n\nStopped by user")
        
        finally:
            self.driver.quit()
            print(f"\nOK Data collection complete for {self.start_date} to {self.end_date}")

print("OK Automation class defined!")

OK Automation class defined!


## Step 4: Configure Date Range

In [6]:
# CONFIGURATION

EXCEL_FILE_PATH = r"C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx"

# DATE RANGE
START_DATE = "1-4-2026"   # شروع کی تاریخ
END_DATE = "2-4-2026"     # آخری تاریخ

print(f"Configuration:")
print(f"  Excel File: {EXCEL_FILE_PATH}")
print(f"  Date Range: {START_DATE} to {END_DATE}")
print(f"\nOK Configuration ready!")

Configuration:
  Excel File: C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx
  Date Range: 1-4-2026 to 2-4-2026

OK Configuration ready!


## Step 5: Start Data Collection

In [7]:
if not os.path.exists(EXCEL_FILE_PATH):
    print(f"ERROR Excel file not found at: {EXCEL_FILE_PATH}")
else:
    print(f"OK Excel file found")
    print(f"\n" + "="*60)
    print(f"COLLECTING DATA FROM {START_DATE} TO {END_DATE}")
    print("="*60 + "\n")
    
    automation = WhatsAppManualLogin(EXCEL_FILE_PATH, START_DATE, END_DATE)
    automation.run_collection()

OK Excel file found

COLLECTING DATA FROM 1-4-2026 TO 2-4-2026


Opening WhatsApp Web...

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
MANUAL LOGIN:
1. Scan QR code with your phone
2. Open your complaint group
3. Press Enter below when ready
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


OK Starting data collection...


Scan #1...
  No new messages
Scan #2...
  No new messages
Scan #3...
  No new messages
Scan #4...
  No new messages
Scan #5...
  No new messages
Scan #6...
  No new messages
Scan #7...
  No new messages
Scan #8...
  No new messages
Scan #9...
  No new messages
Scan #10...
  No new messages

OK Data collection complete for 1-4-2026 to 2-4-2026


## اگلے دن کے لیے:

**Step 4 میں تاریخ بدلیں:**

```python
START_DATE = "2-4-2026"
END_DATE = "2-4-2026"
```

پھر Step 5 دوبارہ چلائیں۔ نیا data Excel میں شامل ہو جائے گا! ✅